# NIG-TimeNet (PyTorch): the PANEL autoregressive candle stream

**Runtime > Run all.** The cross-asset (panel) version of the autoregressive candle stream, in its own
notebook: one shared model over MANY coins on the SAME timestamps. The window is
`CONTEXT_LEN + FUTURE_LEN` candles per coin (48 by default = 32 context + 16 future); the model predicts
every coin's next candle, each coin's own prediction is appended as its next input, and that repeats
candle by candle - N text models generating in lockstep with a shared vocabulary, plus cross-coin
attention so a coin may look at the other coins at the same timestamp (never at their future). The
horizons are free parameters: 4, 8, 16 ... and the EVALUATION horizon may differ from the TRAINING one;
coins are free too: all of them, or an explicit list.

Training follows the same aware recipe as the single-coin notebook: candle-by-candle prediction with
scheduled sampling on DETACHED own predictions (the model learns to consume its own outputs without the
noise becoming a learning signal), and the loss is selectable (`LOSS` below): the default "czar" scores each
return in units of its window's LOCAL volatility (a shock is bounded by the Cauchy kernel, a shrunk-to-zero
prediction on a real move stays expensive, flat noise cannot pay), while "huber" keeps the v1 robust loss
(Huber cap + adaptive outlier weight). The DIRECTION term keeps its full weight in both. Selection
is the validation direction accuracy of the panel-wide rollout, next to the persistence and always-up
trivial baselines and a shuffled-labels control at its honest null.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/ar-stream"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = ()                             # torch/pandas/numpy ship with Colab
SOURCE = "drive"                               # "drive" (your Drive candle files), "binance" (live futures API, synthetic fallback) or "synthetic"
COINS = "all"                                  # 'all', 'all_global' or an explicit tuple: ("BTCUSDT", "ETHUSDT")
DAYS = 120                                     # days of hourly candles per coin (the newest DAYS days are kept)
INTERVAL = "1h"     # timeframe: Drive 1h files are re-aggregated UP to this (2h, 4h, 1d, ...)
RAW_ROOT = "/content/drive/MyDrive"              # the Drive folder holding history_1h/<SYMBOL>.csv (crypto-signal-prediction layout)
CONTEXT_LEN = 32                               # the base window: 32 candles per coin the model conditions on
HORIZON_LEN = 16                               # candles per coin GENERATED while training: 4, 8, 16, ... any number
EVAL_HORIZON = 8                               # candles to SCORE in the rollout: free, independent of HORIZON_LEN
LOSS = "czar"                                   # "czar" (anti-cowardice, shock-bounded) or "huber" (v1)
CHANGES = {}                                   # PanelArStreamSettings overrides, e.g. {"epochs": 20, "cross_layers": 2}
VAL_SHARE, TEST_SHARE = 0.15, 0.15

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data: raw candles -> an aligned panel of gap-free windows
Drive 1h bars are re-aggregated up to INTERVAL (open first / high max / low min / close last, partial trailing
gaps dropped); SOURCE="drive" reads the candle files your Drive
already holds - the crypto-signal-prediction layout `history_1h/<SYMBOL>.csv`) and turned into five scale-free per-candle features.
Each coin's windows are cut only inside gap-free stretches and carry their end timestamps; the panel
builder then keeps only the window positions covered by EVERY coin, producing one aligned
(windows, coins, candles, features) grid. The grid carries `FUTURE_LEN = max(HORIZON_LEN, EVAL_HORIZON)`
future candles per coin so the same data serves a training horizon and a DIFFERENT evaluation horizon.

In [ ]:
from nig_timenet_torch.data.ar_windows import raw_candles
FUTURE_LEN = max(HORIZON_LEN, EVAL_HORIZON)
frames = raw_candles(SOURCE, COINS, DAYS, INTERVAL, root=RAW_ROOT, seed=0)
print(len(frames), "coins,", {k: len(v) for k, v in list(frames.items())[:3]}, "bars ...")

In [ ]:
from nig_timenet_torch.data.ar_windows import build_windows, candle_features, contiguous_runs, fit_volume_median
coins = {}
for name, df in frames.items():
    feats = candle_features(df, fit_volume_median(df))
    coins[name] = build_windows(feats, contiguous_runs(df.index, INTERVAL), CONTEXT_LEN, FUTURE_LEN, index=df.index)
    print(f"{name}: {coins[name].n} windows")

In [ ]:
from nig_timenet_torch.data.ar_windows import panel_windows, split_panel
panel = panel_windows(coins)
data = split_panel(panel, VAL_SHARE, TEST_SHARE)
print(panel["X"].shape, "panel grid (windows, coins, candles, features);", data["coin_names"])

## 3. Train - the panel, candle by candle
`fit_panel_ar_stream` predicts every horizon step for ALL coins at once, mixing each coin's own detached
predictions into its inputs on a rising schedule, scoring the return with the selected LOSS kind ("czar" by default: local-volatility z units,
Cauchy-bounded shocks, anti-cowardice; or "huber": the v1 Huber cap + adaptive outlier weight), keeping the direction term at full weight, and selecting the epoch
by the validation direction accuracy of the panel-wide rollout.

In [ ]:
from nig_timenet_torch.trainer.panel_ar_stream import PanelArStreamSettings, build_panel_ar_stream_model, fit_panel_ar_stream
settings = PanelArStreamSettings(context_len=CONTEXT_LEN, horizon_len=HORIZON_LEN, device=EXPECT_DEVICE, loss=LOSS, **CHANGES)
model = build_panel_ar_stream_model(settings, n_features=data["train_X"].shape[3],
                                    n_coins=data["train_X"].shape[1])
history = fit_panel_ar_stream(model, data["train_X"], data["train_Y"], data["train_dirs"],
                              data["val_X"], data["val_Y"], data["val_dirs"], settings)
print("best epoch", history["best_epoch"], "val direction accuracy", round(history["best_val_dir_acc"], 4))

## 4. Panel rollout: direction first, at a FREE horizon
The test is the honest one: the model eats only its own predictions for the whole rollout - every coin
generating `EVAL_HORIZON` candles in lockstep, which may differ from the horizon it was trained on.
Direction accuracy is reported overall and per step, next to the two trivial baselines (persistence
sign, always-up): the model only counts when it beats them.

In [ ]:
from nig_timenet_torch.eval.ar_stream import panel_rollout_metrics
report = panel_rollout_metrics(model, data["test_X"], data["test_Y"], data["test_dirs"],
                               data["test_last_ret"], EXPECT_DEVICE, horizon_len=EVAL_HORIZON)
print("direction", round(report["direction_accuracy"], 4), "persistence", round(report["persistence_accuracy"], 4),
      "always-up", round(report["always_up_accuracy"], 4), "gap", round(report["trivial_gap"], 4))
print("per step:", [round(a, 3) for a in report["direction_accuracy_per_step"]])

## 5. Controls and self-checks
The shuffled-labels control must land at its honest null - the majority agreement of a biased model,
~50% for a balanced one: anything clearly higher means the scoring leaks and any claim is void. The shape checks pin the panel contract (windows x coins x 32 + FUTURE_LEN).

In [ ]:
import numpy as np
from nig_timenet_torch.eval.ar_stream import panel_direction_control
control = panel_direction_control(model, data["test_X"], data["test_dirs"], EXPECT_DEVICE, horizon_len=EVAL_HORIZON)
print("shuffled control", round(control["shuffled_direction_accuracy"], 4), "expected ~", round(control["expected"], 4))
assert data["train_X"].shape[2] == CONTEXT_LEN and data["train_Y"].shape[2] == FUTURE_LEN
assert data["train_X"].shape[1] == len(data["coin_names"]) == panel["X"].shape[1]
assert 1 <= EVAL_HORIZON <= FUTURE_LEN and set(np.unique(data["train_dirs"])) <= {-1.0, 0.0, 1.0}
assert abs(control["shuffled_direction_accuracy"] - control["expected"]) < 0.1
print("self-checks passed")

## 6. Save the run: the prediction tables land on Drive
Every (window, coin, step) prediction of the VAL and TEST panel rollouts - the model eating only its
own candles, every coin in lockstep - plus the history, report and settings are written under
`DRIVE_RUNS`: exactly the files the evaluation-battery notebook (`ar_stream_battery.ipynb`)
auto-detects and judges.


In [ ]:
from datetime import datetime
import numpy as np
from nig_timenet_torch.eval.ar_stream import panel_rollout_predictions, dump_run
preds = {s: panel_rollout_predictions(model, data[f"{s}_X"], data[f"{s}_Y"], data[f"{s}_dirs"],
                                     data[f"{s}_last_ret"], EXPECT_DEVICE, horizon_len=EVAL_HORIZON,
                                     coin_names=np.array(data["coin_names"]),
                                     end_times=data[f"{s}_end_times"])
         for s in ("val", "test")}
run = dump_run(DRIVE_RUNS, "panel-ar-stream-" + datetime.now().strftime("%Y%m%d-%H%M%S"),
               preds, history, report, settings)
print("run saved:", run)
